In [1]:
import pickle
import torch
import pandas as pd

In [2]:
import pickle

with open(
    "../models/semantic_neighbors.pkl",
    "rb"
) as f:
    semantic_neighbors = pickle.load(f)

print(type(semantic_neighbors))
print(len(semantic_neighbors))

<class 'dict'>
373


In [3]:
import os

print(os.path.exists("../data/processed/train_df.csv"))
print(os.path.exists("../data/processed/test_df.csv"))
print(os.path.exists("../models/item_similarity_df.pkl"))

True
True
True


In [4]:
train_df = pd.read_csv(
    "../data/processed/train_df.csv"
)

test_df = pd.read_csv(
    "../data/processed/test_df.csv"
)

with open(
    "../models/item_similarity_df.pkl",
    "rb"
) as f:
    item_similarity_df = pickle.load(f)

print(train_df.shape)
print(test_df.shape)
print(item_similarity_df.shape)

(101908, 5)
(6197, 5)
(9009, 9009)


In [5]:
import pandas as pd
import pickle

train_df = pd.read_csv(
    "../data/processed/train_df.csv"
)

test_df = pd.read_csv(
    "../data/processed/test_df.csv"
)

with open(
    "../models/item_similarity_df.pkl",
    "rb"
) as f:
    item_similarity_df = pickle.load(f)

In [6]:
print(train_df.shape)
print(test_df.shape)
print(item_similarity_df.shape)

(101908, 5)
(6197, 5)
(9009, 9009)


In [7]:
import os

path = "../models/item_similarity_df.pkl"

print("Exists:", os.path.exists(path))
print("Size:", os.path.getsize(path))

Exists: True
Size: 649369281


In [8]:
import torch

config = torch.load(
    "../models/bpr_config.pth",
    weights_only=False
)

print(config)

{'n_users': np.int64(6436), 'n_books': np.int64(9009), 'embedding_dim': 50}


In [9]:
import torch
import torch.nn as nn

class BPRModel(nn.Module):

    def __init__(self, n_users, n_books, embedding_dim=50):

        super().__init__()

        self.user_embedding = nn.Embedding(
            n_users,
            embedding_dim
        )

        self.book_embedding = nn.Embedding(
            n_books,
            embedding_dim
        )

    def forward(
        self,
        users,
        pos_items,
        neg_items
    ):

        user_emb = self.user_embedding(users)

        pos_emb = self.book_embedding(pos_items)
        neg_emb = self.book_embedding(neg_items)

        pos_scores = (
            user_emb * pos_emb
        ).sum(dim=1)

        neg_scores = (
            user_emb * neg_emb
        ).sum(dim=1)

        return pos_scores, neg_scores

In [10]:
config = torch.load(
    "../models/bpr_config.pth",
    weights_only=False
)

model = BPRModel(
    n_users=int(config["n_users"]),
    n_books=int(config["n_books"]),
    embedding_dim=int(config["embedding_dim"])
)

In [11]:
model.load_state_dict(
    torch.load(
        "../models/bpr_model.pth",
        weights_only=True
    )
)

model.eval()

print("BPR loaded successfully")

BPR loaded successfully


In [12]:
print(
    model.user_embedding.weight.shape
)

print(
    model.book_embedding.weight.shape
)

torch.Size([6436, 50])
torch.Size([9009, 50])


In [13]:
user_history = (
    train_df
    .groupby("user_idx")["book_idx"]
    .apply(set)
    .to_dict()
)

print(len(user_history))

6197


In [14]:
book_idx_to_isbn = (
    train_df[
        ["book_idx", "ISBN"]
    ]
    .drop_duplicates()
    .set_index("book_idx")["ISBN"]
    .to_dict()
)

isbn_to_book_idx = {
    v: k
    for k, v in book_idx_to_isbn.items()
}

In [15]:
print(len(user_history))
print(len(book_idx_to_isbn))

6197
9009


In [16]:
import torch
import numpy as np

def recommend_bpr(
    user_idx,
    model,
    user_history,
    top_k=10
):

    model.eval()

    with torch.no_grad():

        user_vec = (
            model.user_embedding.weight[
                user_idx
            ]
        )

        book_vecs = (
            model.book_embedding.weight
        )

        scores = torch.matmul(
            book_vecs,
            user_vec
        )

        scores = scores.cpu().numpy()

    seen_books = user_history.get(
        user_idx,
        set()
    )

    scores[list(seen_books)] = -np.inf

    top_books = np.argsort(
        scores
    )[::-1][:top_k]

    return [
        (
            int(book_idx),
            float(scores[book_idx])
        )
        for book_idx in top_books
    ]

In [17]:
sample_user = train_df[
    "user_idx"
].iloc[0]

print(sample_user)

0


In [18]:
recs = recommend_bpr(
    sample_user,
    model,
    user_history,
    top_k=10
)

print(recs)

[(366, 1.1541355848312378), (431, 0.9889606237411499), (496, 0.9762865900993347), (683, 0.96480393409729), (1833, 0.9269967079162598), (502, 0.8921451568603516), (1619, 0.8710893392562866), (1297, 0.8250557780265808), (626, 0.7830356955528259), (981, 0.7641429305076599)]


In [19]:
sample_user = train_df["user_idx"].iloc[0]

recs = recommend_bpr(
    sample_user,
    model,
    user_history,
    top_k=10
)

print(len(recs))
print(recs[:5])

10
[(366, 1.1541355848312378), (431, 0.9889606237411499), (496, 0.9762865900993347), (683, 0.96480393409729), (1833, 0.9269967079162598)]


In [20]:
print(type(item_similarity_df))
print(item_similarity_df.shape)

print(item_similarity_df.index[:5])
print(item_similarity_df.columns[:5])

<class 'pandas.DataFrame'>
(9009, 9009)
Index([0, 1, 2, 3, 4], dtype='int64', name='book_idx')
Index([0, 1, 2, 3, 4], dtype='int64', name='book_idx')


In [21]:
from collections import defaultdict
import numpy as np
import pandas as pd

def recommend_cf(
    user_id,
    train_df,
    item_similarity_df,
    top_n=100,
    k=10
):

    user_ratings = train_df[
        train_df["user_idx"] == user_id
    ]

    scores = {}

    for _, row in user_ratings.iterrows():

        book = row["book_idx"]
        rating = row["Rating"]

        neighbors = (
            item_similarity_df[book]
            .sort_values(ascending=False)
            .iloc[1:k+1]
        )

        for sim_book, sim_score in neighbors.items():

            scores[sim_book] = (
                scores.get(sim_book, 0)
                + sim_score * rating
            )

    already_read = set(
        user_ratings["book_idx"]
    )

    recommendations = (
        pd.Series(scores)
        .drop(
            labels=already_read,
            errors="ignore"
        )
        .sort_values(
            ascending=False
        )
        .head(top_n)
    )

    return list(
        zip(
            recommendations.index.tolist(),
            recommendations.values.tolist()
        )
    )

In [22]:
sample_user = train_df["user_idx"].iloc[0]

cf_recs = recommend_cf(
    sample_user,
    train_df,
    item_similarity_df,
    top_n=10,
    k=10
)

print(len(cf_recs))
print(cf_recs[:5])

10
[(4458, 2.415546350523316), (8463, 2.2792589017368887), (8616, 2.2224491996215394), (3899, 2.0400775810920493), (5253, 1.9397066178026678)]


In [23]:
print("semantic_neighbors" in globals())

True


In [24]:
print(type(semantic_neighbors))
print(len(semantic_neighbors))

<class 'dict'>
373


In [25]:
book_idx_to_isbn = (
    train_df[
        ["book_idx", "ISBN"]
    ]
    .drop_duplicates()
    .set_index("book_idx")["ISBN"]
    .to_dict()
)

isbn_to_book_idx = {
    isbn: idx
    for idx, isbn in book_idx_to_isbn.items()
}

print(len(book_idx_to_isbn))
print(len(isbn_to_book_idx))

9009
9009


In [26]:
matched = 0

for isbn in semantic_neighbors.keys():

    if isbn in isbn_to_book_idx:
        matched += 1

print("Matched:", matched)
print("Total Semantic Books:", len(semantic_neighbors))

Matched: 122
Total Semantic Books: 373


In [27]:
from collections import defaultdict

def recommend_content(
    user_idx,
    user_history,
    book_idx_to_isbn,
    isbn_to_book_idx,
    semantic_neighbors,
    top_n=10
):

    scores = defaultdict(float)

    seen_books = user_history.get(
        user_idx,
        set()
    )

    for book_idx in seen_books:

        isbn = book_idx_to_isbn.get(book_idx)

        if isbn not in semantic_neighbors:
            continue

        for neighbor_isbn, sim_score in semantic_neighbors[isbn]:

            if neighbor_isbn not in isbn_to_book_idx:
                continue

            neighbor_idx = isbn_to_book_idx[
                neighbor_isbn
            ]

            if neighbor_idx in seen_books:
                continue

            scores[neighbor_idx] += sim_score

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked[:top_n]

In [28]:
content_recs = recommend_content(
    user_idx=0,
    user_history=user_history,
    book_idx_to_isbn=book_idx_to_isbn,
    isbn_to_book_idx=isbn_to_book_idx,
    semantic_neighbors=semantic_neighbors,
    top_n=10
)

print(content_recs)

[]


In [29]:
sample_user = 0

seen_books = user_history[sample_user]

metadata_books = 0

for book_idx in seen_books:

    isbn = book_idx_to_isbn.get(book_idx)

    if isbn in semantic_neighbors:
        metadata_books += 1

print("Seen books:", len(seen_books))
print("Books with metadata:", metadata_books)

Seen books: 2
Books with metadata: 0


In [30]:
for user_idx, books in user_history.items():

    count = 0

    for book_idx in books:

        isbn = book_idx_to_isbn.get(book_idx)

        if isbn in semantic_neighbors:
            count += 1

    if count > 0:

        print(
            "User:",
            user_idx,
            "Metadata books:",
            count
        )

        break

User: 2 Metadata books: 1


In [31]:
content_recs = recommend_content(
    user_idx=1,
    user_history=user_history,
    book_idx_to_isbn=book_idx_to_isbn,
    isbn_to_book_idx=isbn_to_book_idx,
    semantic_neighbors=semantic_neighbors,
    top_n=10
)

print(content_recs)

[]


In [32]:
def normalize_scores(recommendations):

    if len(recommendations) == 0:
        return {}

    scores = [score for _, score in recommendations]

    min_score = min(scores)
    max_score = max(scores)

    if max_score == min_score:

        return {
            item: 1.0
            for item, _ in recommendations
        }

    return {
        item: (score - min_score) /
              (max_score - min_score)
        for item, score in recommendations
    }

In [33]:
cf_test = normalize_scores(cf_recs)

list(cf_test.items())[:5]

[(4458, 1.0),
 (8463, 0.7667728214412546),
 (8616, 0.6695550170351513),
 (3899, 0.3574645170106128),
 (5253, 0.18570081620284531)]

In [34]:
books = pd.read_csv(
    "../data/processed/books_master.csv",
    low_memory=False
)

print(books.shape)

(271360, 13)


In [35]:
books_cf = books[
    books["ISBN"].isin(
        train_df["ISBN"].unique()
    )
].copy()

print(len(books_cf))

9009


In [36]:
book_author = (
    books_cf[
        ["ISBN", "Book-Author"]
    ]
    .drop_duplicates()
)

In [37]:
book_author["book_idx"] = (
    book_author["ISBN"]
    .map(isbn_to_book_idx)
)

book_author = book_author.dropna()

print(book_author.shape)

(9009, 3)


In [38]:
author_groups = (
    book_author
    .groupby("Book-Author")["book_idx"]
    .apply(list)
    .to_dict()
)

print(len(author_groups))

3142


In [39]:
author_sizes = [
    len(books)
    for books in author_groups.values()
]

print("Authors:", len(author_sizes))
print("Avg books/author:", sum(author_sizes)/len(author_sizes))
print("Max books by one author:", max(author_sizes))

Authors: 3142
Avg books/author: 2.867281985996181
Max books by one author: 147


In [40]:
author_neighbors = {}

for author, books in author_groups.items():

    for book in books:

        neighbors = []

        for other_book in books:

            if other_book != book:

                neighbors.append(
                    (int(other_book), 1.0)
                )

        author_neighbors[int(book)] = neighbors

In [41]:
print(len(author_neighbors))

9009


In [42]:
from collections import defaultdict

def recommend_author(
    user_idx,
    user_history,
    author_neighbors,
    top_n=100
):

    scores = defaultdict(float)

    seen_books = user_history.get(
        user_idx,
        set()
    )

    for book_idx in seen_books:

        if book_idx not in author_neighbors:
            continue

        for neighbor_idx, score in author_neighbors[book_idx]:

            if neighbor_idx in seen_books:
                continue

            scores[neighbor_idx] += score

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked[:top_n]

In [43]:
sample_user = next(iter(user_history.keys()))

author_recs = recommend_author(
    sample_user,
    user_history,
    author_neighbors,
    top_n=10
)

print(author_recs[:10])

[(1796, 1.0), (6266, 1.0), (1563, 1.0), (4459, 1.0), (4458, 1.0), (5987, 1.0), (8616, 1.0)]


In [44]:
sample_user = next(iter(user_history.keys()))

seen_books = user_history[sample_user]

print("Books seen:", len(seen_books))

Books seen: 2


In [45]:
for book_idx in list(seen_books)[:10]:

    author = (
        book_author.loc[
            book_author["book_idx"] == book_idx,
            "Book-Author"
        ]
        .iloc[0]
    )

    print(author)

CARL HIAASEN
Eoin Colfer


In [46]:
author_sizes = pd.Series(
    {
        author: len(books)
        for author, books in author_groups.items()
    }
)

print(author_sizes.describe())

count    3142.000000
mean        2.867282
std         5.786052
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max       147.000000
dtype: float64


In [47]:
print(
    (author_sizes > 1).sum()
)

1239


In [48]:
for user_idx, books_seen in user_history.items():

    found = False

    for book_idx in books_seen:

        if len(author_neighbors[book_idx]) > 0:

            found = True
            break

    if found:

        print("User:", user_idx)
        break

User: 0


In [49]:
author_recs = recommend_author(
    user_idx,
    user_history,
    author_neighbors,
    top_n=10
)

print(author_recs)

[(1796, 1.0), (6266, 1.0), (1563, 1.0), (4459, 1.0), (4458, 1.0), (5987, 1.0), (8616, 1.0)]


In [50]:
from collections import defaultdict

def recommend_content_combined(
    user_idx,
    user_history,
    author_neighbors,
    semantic_neighbors,
    book_idx_to_isbn,
    isbn_to_book_idx,
    top_n=100
):

    scores = defaultdict(float)

    # ---------- Author Part ----------
    seen_books = user_history.get(
        user_idx,
        set()
    )

    for book_idx in seen_books:

        for neighbor_idx, score in author_neighbors.get(
            book_idx,
            []
        ):

            if neighbor_idx in seen_books:
                continue

            scores[neighbor_idx] += 0.8 * score

    # ---------- Semantic Part ----------
    for book_idx in seen_books:

        isbn = book_idx_to_isbn.get(book_idx)

        if isbn not in semantic_neighbors:
            continue

        for neighbor_isbn, score in semantic_neighbors[isbn]:

            if neighbor_isbn not in isbn_to_book_idx:
                continue

            neighbor_idx = isbn_to_book_idx[
                neighbor_isbn
            ]

            if neighbor_idx in seen_books:
                continue

            scores[neighbor_idx] += 0.2 * score

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked[:top_n]

In [51]:
content_recs = recommend_content_combined(
    sample_user,
    user_history,
    author_neighbors,
    semantic_neighbors,
    book_idx_to_isbn,
    isbn_to_book_idx,
    top_n=10
)

print(content_recs[:10])

[(1796, 0.8), (6266, 0.8), (1563, 0.8), (4459, 0.8), (4458, 0.8), (5987, 0.8), (8616, 0.8)]


In [52]:
print(sample_user)

0


In [53]:
author_recs = recommend_author(
    sample_user,
    user_history,
    author_neighbors,
    top_n=10
)

print(author_recs[:10])

[(1796, 1.0), (6266, 1.0), (1563, 1.0), (4459, 1.0), (4458, 1.0), (5987, 1.0), (8616, 1.0)]


In [54]:
good_user = None

for user_idx, books_seen in user_history.items():

    if any(
        len(author_neighbors[book]) > 0
        for book in books_seen
    ):
        good_user = user_idx
        break

print(good_user)

0


In [55]:
author_recs = recommend_author(
    good_user,
    user_history,
    author_neighbors,
    top_n=10
)

print(author_recs[:10])

[(1796, 1.0), (6266, 1.0), (1563, 1.0), (4459, 1.0), (4458, 1.0), (5987, 1.0), (8616, 1.0)]


In [56]:
content_recs = recommend_content_combined(
    good_user,
    user_history,
    author_neighbors,
    semantic_neighbors,
    book_idx_to_isbn,
    isbn_to_book_idx,
    top_n=10
)

print(content_recs[:10])


[(1796, 0.8), (6266, 0.8), (1563, 0.8), (4459, 0.8), (4458, 0.8), (5987, 0.8), (8616, 0.8)]


In [57]:
cf_recs = recommend_cf(
    user_idx,
    train_df,
    item_similarity_df,
    top_n=100,
    k=10
)

bpr_recs = recommend_bpr(
    user_idx,
    model,
    user_history,
    top_k=100
)

content_recs = recommend_content_combined(
    user_idx,
    user_history,
    author_neighbors,
    semantic_neighbors,
    book_idx_to_isbn,
    isbn_to_book_idx,
    top_n=100
)

In [143]:
from collections import defaultdict

def hybrid_recommend(
    user_idx,
    train_df,
    user_history,
    item_similarity_df,
    model,
    author_neighbors,
    semantic_neighbors,
    book_idx_to_isbn,
    isbn_to_book_idx,
    top_n=10,
    cf_weight=0.3,
    bpr_weight=0.35,
    content_weight=0.35
):

    cf_recs = recommend_cf(
        user_idx,
        train_df,
        item_similarity_df,
        top_n=100,
        k=10
    )

    bpr_recs = recommend_bpr(
        user_idx,
        model,
        user_history,
        top_k=100
    )

    content_recs = recommend_content_combined(
        user_idx,
        user_history,
        author_neighbors,
        semantic_neighbors,
        book_idx_to_isbn,
        isbn_to_book_idx,
        top_n=100
    )

    cf_scores = normalize_scores(cf_recs)
    bpr_scores = normalize_scores(bpr_recs)
    content_scores = normalize_scores(content_recs)

    final_scores = defaultdict(float)

    all_books = (
        set(cf_scores.keys()) |
        set(bpr_scores.keys()) |
        set(content_scores.keys())
    )

    for book in all_books:

        final_scores[book] += (
            cf_weight *
            cf_scores.get(book, 0)
        )

        final_scores[book] += (
            bpr_weight *
            bpr_scores.get(book, 0)
        )

        final_scores[book] += (
            content_weight *
            content_scores.get(book, 0)
        )

    ranked = sorted(
        final_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked[:top_n]

In [144]:
hybrid_recs = hybrid_recommend(
    good_user,
    train_df,
    user_history,
    item_similarity_df,
    model,
    author_neighbors,
    semantic_neighbors,
    book_idx_to_isbn,
    isbn_to_book_idx,
    top_n=10,
    cf_weight=0,
    bpr_weight=1,
    content_weight=0
)

print(hybrid_recs[:10])

[(366, 1.0), (431, 0.7457895572102291), (496, 0.7262837454167511), (683, 0.7086115083693312), (1833, 0.6504247732570353), (502, 0.5967869327068528), (1619, 0.5643812419102275), (1297, 0.4935338694866853), (626, 0.42886339097578136), (981, 0.39978671896715107)]


In [145]:
len(hybrid_recs)

10

In [146]:
hits = 0

for _, row in test_df.iterrows():

    user_idx = row["user_idx"]
    true_book = row["book_idx"]

    recs = hybrid_recommend(
        user_idx,
        train_df,
        user_history,
        item_similarity_df,
        model,
        author_neighbors,
        semantic_neighbors,
        book_idx_to_isbn,
        isbn_to_book_idx,
        top_n=10
    )

    recommended_books = [
        book
        for book, _
        in recs
    ]

    if true_book in recommended_books:
        hits += 1

hit10 = hits / len(test_df)

print("Hits:", hits)
print("Hit@10:", hit10)

Hits: 521
Hit@10: 0.08407293851863805


In [147]:
import pickle

hybrid_config = {
    "cf_weight": 0.30,
    "bpr_weight": 0.35,
    "content_weight": 0.35,
    "hit10": 0.08407293851863805
}

with open(
    "../models/hybrid_config.pkl",
    "wb"
) as f:
    pickle.dump(hybrid_config, f)

print("Hybrid configuration saved!")
    

Hybrid configuration saved!


In [148]:
results = {
    "CF": 0.059,
    "BPR": 0.052,
    "Content": 0.049,
    "Hybrid": 0.08407
}